In [ ]:
#| default_exp magic/turns

In [ ]:
#| export
from __future__ import annotations
from datetime import datetime
import numpy as np
from dataclasses import dataclass, field
import io, base64, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import NamedTuple
from enum import Enum
from collections import defaultdict
import pandas as pd
import random
import math
import uuid
import httpx

import networkx as nx

from monsterui.all import *
from fastcore.basics import patch

In [ ]:
#| export
from fasthtml.common import *
from fasthtml.jupyter import *
from fasthtml.common import *

In [ ]:
#| export
from HexMagic.primitives import HexTouchMap, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.primitives import MapRect, MapPath,MapCord,  MapSize, HexRegion
from HexMagic.styles import StyleCSS, SVGBuilder,SVGLayer, SVGPatternLoader, preview, app, rt, StyleDemo
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import Terrain, HexGrid, HexPosition
from HexMagic.overlay import CreamOverlay, RiverOverlay,OceanWaveOverlay


In [ ]:
#| export
from HexMagic.magic.pieces import GameParts, Piece, Squad, PieceType
from HexMagic.magic.food import FOOD_TABLE, FoodStats
 # Build main map
from HexMagic.magic.pieces import PieceOverlay, LanternOverlay, SquadNamesOverlay
from HexMagic.magic.moves import MoveError, Move, MoveLog, PurchasePreview, Placement, Purchase, Rotation, PURCHASE_COST
from HexMagic.magic.network import SquadFlow, Squad

In [ ]:
#| export
@dataclass 
class GameRecord:
    _df: pd.DataFrame = field(default_factory=lambda: pd.DataFrame(columns=[
        'turn', 'squad', 'is_player', 'alive',
        'king_pantry', 'total_harvest', 'total_diet', 'net_food',
        'drain_dealt', 'drain_taken',
        'pieces_placed', 'pieces_available',
        'bottlenecks', 'king_connected', 'isolated',
        'sight_coverage',
        'target_type', 'stance',
        'drain_by_enemy',   # dict: {enemy_squad_name: total_drain}
    ]))
    
    def snapshot(self, turn: int, parts: GameParts, player_squad: Squad,
                 summaries: list[dict] | None = None):
        # Index summaries by squad name for fast lookup
        summary_map = {s['squad']: s for s in (summaries or [])}
        
        rows = []
        for sq in parts.squads:
            placed = [p for p in sq.pieces if p.place is not None]
            king = next((p for p in placed if p.type == PieceType.KING), None)
            
            # Food
            harvest = sum(int(p.food.harvest * p.place.harvest) for p in placed)
            diet = sum(p.food.diet for p in placed)
            
            # Drain — from authoritative summary if available
            s = summary_map.get(sq.name, {})
            drain_dealt = s.get('food_drain_dealt', 0)
            drain_taken = sum(
                edge[2] for other_s in summary_map.values()
                for edge in other_s.get('drain_edges', [])
                if edge[1] == sq.name
            )
            
            # Aggregate drain_edges → {enemy_name: total}
            drain_by_enemy = {}
            for piece_name, enemy_name, amount in s.get('drain_edges', []):
                drain_by_enemy[enemy_name] = drain_by_enemy.get(enemy_name, 0) + amount

            # Network
            G = sq.to_nx()
            U = G.to_undirected(as_view=True) if len(G) > 1 else nx.Graph()
            bn = len(list(nx.articulation_points(U))) if len(G) > 1 else 0
            king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
            king_comp = len(nx.node_connected_component(U, king_id)) if king_id and len(G) > 1 else len(placed)
            iso = len(sq.isolated_pieces())
            
            # Sight
            visible = set()
            for p in placed: visible |= p.sight().hexes
            
            target = getattr(sq, '_target', None)
            
            rows.append(dict(
                turn=turn, squad=sq.name,
                is_player=sq is player_squad,
                alive=sq.king_alive(),
                king_pantry=king.place.pantry if king else 0,
                total_harvest=harvest,
                total_diet=diet,
                net_food=harvest - diet,
                drain_dealt=drain_dealt,
                drain_taken=drain_taken,
                pieces_placed=len(placed),
                pieces_available=sum(1 for p in sq.pieces if p.place is None),
                bottlenecks=bn,
                king_connected=king_comp,
                isolated=iso,
                sight_coverage=len(visible),
                target_type=target.type.value if target else 'none',
                stance=getattr(sq, '_stance', 'expand'),
                drain_by_enemy=drain_by_enemy,
            ))
        
        self._df = pd.concat([self._df, pd.DataFrame(rows)], ignore_index=True)

    
    @property
    def df(self) -> pd.DataFrame:
        return self._df
    
    def for_squad(self, name: str) -> pd.DataFrame:
        return self._df[self._df.squad == name]
    
    def latest(self) -> pd.DataFrame:
        if self._df.empty: return self._df
        return self._df[self._df.turn == self._df.turn.max()]
    
    def sparkline_data(self, col: str) -> dict[str, list]:
        """Dict of squad→values for any column, ready for plotting."""
        return {
            name: grp.sort_values('turn')[col].tolist()
            for name, grp in self._df.groupby('squad')
        }


## Turn Processor

In [ ]:
#| export
@dataclass
class TurnProcessor:
    parts: GameParts
    squads: list[Squad]
    log: MoveLog = None
    record: GameRecord = None      # ← add this
    turn: int = 0
    game_over: bool = False
    winner: Squad | None = None
    player_squad: Squad | None = None  # optional — None for pure AI games
    summaries_by_turn: dict[int, list[dict]] = field(default_factory=dict)  # ← new

    def __post_init__(self):
        if self.log is None:
            self.log = MoveLog(self.parts)
        if self.record is None:
            self.record = GameRecord()
    
    def _king(self, squad: Squad) -> Piece | None:
        return next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
    

    
    def _check_game_over(self):
        alive = [s for s in self.squads if s.king_alive()]
        if len(alive) <= 1:
            self.game_over = True
            self.winner = alive[0] if alive else None
    
    def status(self) -> pd.DataFrame:
        rows = []
        for s in self.squads:
            king = self._king(s)
            placed = [p for p in s.pieces if p.place is not None]
            G = s.to_nx()
            king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
            bottlenecks = len(list(nx.articulation_points(G.to_undirected(as_view=True)))) if len(G) > 1 else 0
            rows.append({
                'squad': s.name,
                'king_pantry': king.place.pantry if king else 0,
                'placed': len(placed),
                'bottlenecks': bottlenecks,
                'king': '👑' if s.king_alive() else '💀',
            })
        return pd.DataFrame(rows)


In [ ]:
#| export
@patch
def _placement_cost(self:TurnProcessor, squad: Squad, move: Move, summary: dict):
    if move.placement is None: return
    piece = move.placement.piece
    cost = piece.food.cost
    king = self._king(squad)
    if king and piece.type != PieceType.KING:
        king.place.pantry -= cost
    piece.place.pantry = int(cost * 0.75)
    summary['placement_cost'] = cost
    summary['placement_pantry'] = piece.place.pantry

@patch
def _food_cycle(self:TurnProcessor, squad: Squad, summary: dict):
    king = self._king(squad)
    placed = [p for p in squad.pieces if p.place is not None]
    for p in placed:
        earned = int(p.food.harvest * p.place.harvest)
        king.place.pantry += earned
        summary['food_harvested'] += earned
    for p in placed:
        king.place.pantry -= p.food.diet
        summary['food_diet'] += p.food.diet

@patch
def _starvation_priority(self:TurnProcessor, p: Piece, king_hp) -> tuple:
    RANK = {PieceType.PAWN: 0, PieceType.KNIGHT: 1,
            PieceType.BISHOP: 2, PieceType.QUEEN: 3}
    return (p.place.pantry,
            RANK.get(p.type, 0),
            -(p.place.location.distance(king_hp) if king_hp else 0))
@patch
def _apply_starvation(self:TurnProcessor, squad: Squad, summary: dict):
    king = self._king(squad)
    king_hp = king.place.location if king else None
    placed = [p for p in squad.pieces if p.place is not None]
    starving = sorted(
        [p for p in placed if p.place.pantry < 0 and p.type != PieceType.KING],
        key=lambda p: self._starvation_priority(p, king_hp)
    )
    for p in starving:
        p.place = None
        summary['starved'].append(p.name)

@patch
def _connectivity_snapshot(self:TurnProcessor, squad: Squad, summary: dict):
    G = squad.to_nx()
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if king_id:
        U = G.to_undirected(as_view=True)
        connected = sum(1 for n in G.nodes if nx.has_path(U, n, king_id))
        summary['connected'] = connected
        summary['disconnected'] = len(G) - connected




In [ ]:
#| export
class DrainEdge(NamedTuple):
    piece: str
    target_squad: str
    amount: int


In [ ]:
#| export
@patch
def _apply_drain(self: TurnProcessor, squad: Squad, summary: dict):
    placed = [p for p in squad.pieces if p.place is not None]
    drained_squads: dict[str, Squad] = {}  # name → squad
    summary.setdefault('drain_edges', [])
    
    for p in placed:
        cone = p.sight().hexes
        for enemy_sq in self.squads:
            if enemy_sq is squad: continue
            for enemy in enemy_sq.pieces:
                if enemy.place is None: continue
                if self.parts.hp2i(enemy.place.location) in cone:
                    drain = int(p.food.drain)
                    enemy_king = self._king(enemy_sq)
                    if enemy_king:
                        enemy_king.place.pantry -= drain
                        summary['food_drain_dealt'] += drain
                        summary['drain_edges'].append(DrainEdge(p.name, enemy_sq.name, drain))
                        drained_squads[enemy_sq.name] = enemy_sq

    for enemy_sq in drained_squads.values():
        enemy_king = self._king(enemy_sq)
        if enemy_king and enemy_king.place.pantry <= 0:
            enemy_king.place.pantry = 0
            enemy_sq.kill()
            summary.setdefault('kills', []).append(enemy_sq.name)


@patch
def resolve_move(self: TurnProcessor, squad: Squad, move: Move) -> dict:
    if self.game_over: raise MoveError("Game is over")
    if not squad.king_alive(): raise MoveError(f"{squad.name} is eliminated")

    king = self._king(squad)
    summary = {
        'squad': squad.name, 'turn': self.turn,
        'placement_cost': 0, 'placement_pantry': 0,
        'food_harvested': 0.0, 'food_diet': 0.0,
        'food_drain_dealt': 0.0, 'starved': [], 'expired': [],
        'connected': 0, 'disconnected': 0,
        'king_pantry_before': king.place.pantry if king else 0,
        'king_pantry_after': 0, 'pantries': {},
        'game_over': False, 'eliminated': None,
    }

    self.log.record(squad, move)
    if move.purchase is not None:
        summary['purchase_type'] = move.purchase.piece_type.value
        summary['purchase_cost'] = PURCHASE_COST[move.purchase.piece_type]

    if move.placement:
        summary['placed_name'] = move.placement.piece.name
        summary['placed_type'] = move.placement.piece.type.value



    self._placement_cost(squad, move, summary)
    summary['expired'] = [p.name for p in squad.tick_lifespan()]
    self._food_cycle(squad, summary)
    self._apply_drain(squad, summary)
    squad.resolve_flow()
    self._connectivity_snapshot(squad, summary)
    self._apply_starvation(squad, summary)

    summary['pantries'] = {p.name: p.place.pantry 
                           for p in squad.pieces if p.place is not None}
    self._check_elimination(squad, summary)
    summary['king_pantry_after'] = (king.place.pantry 
                                    if (king := self._king(squad)) and king.place else 0)
    self._check_game_over()
    self.record.snapshot(self.turn, self.parts, self.player_squad, summaries=[summary])
    return summary

In [ ]:
#| export
@patch
def resolve_move(self: TurnProcessor, squad: Squad, move: Move) -> dict:
    if self.game_over: raise MoveError("Game is over")
    if not squad.king_alive(): raise MoveError(f"{squad.name} is eliminated")

    king = self._king(squad)
    summary = {
        'squad': squad.name, 'turn': self.turn,
        'placement_cost': 0, 'placement_pantry': 0,
        'food_harvested': 0.0, 'food_diet': 0.0,
        'food_drain_dealt': 0.0, 'starved': [], 'expired': [],
        'connected': 0, 'disconnected': 0,
        'king_pantry_before': king.place.pantry if king else 0,
        'king_pantry_after': 0, 'pantries': {},
        'game_over': False, 'eliminated': None,
    }

    self.log.record(squad, move)
    if move.purchase is not None:
        summary['purchase_type'] = move.purchase.piece_type.value
        summary['purchase_cost'] = PURCHASE_COST[move.purchase.piece_type]

    if move.placement:
        summary['placed_name'] = move.placement.piece.name
        summary['placed_type'] = move.placement.piece.type.value



    self._placement_cost(squad, move, summary)
    summary['expired'] = [p.name for p in squad.tick_lifespan()]
    self._food_cycle(squad, summary)
    self._apply_drain(squad, summary)
    squad.resolve_flow()
    self._connectivity_snapshot(squad, summary)
    self._apply_starvation(squad, summary)

    summary['pantries'] = {p.name: p.place.pantry 
                           for p in squad.pieces if p.place is not None}
    self._check_elimination(squad, summary)
    summary['king_pantry_after'] = (king.place.pantry 
                                    if (king := self._king(squad)) and king.place else 0)
    self._check_game_over()
    
    
    # Only stash if called standalone (not from play_round)
    if not getattr(self, '_in_round', False):
        self.summaries_by_turn.setdefault(self.turn, []).append(summary)
    
    self.record.snapshot(self.turn, self.parts, self.player_squad, summaries=[summary])
    return summary

In [ ]:
#| export
@patch
def _check_elimination(self: TurnProcessor, squad: Squad, summary: dict):
    king = self._king(squad)
    if not squad.king_alive():
        squad.kill()
        summary['eliminated'] = squad.name
        return
    if king and king.place.pantry <= 0:
        king.place.pantry = 0  # clamp
        squad.kill()
        summary['eliminated'] = squad.name


In [ ]:
#| export
@patch
def setup_game(self: GameParts, n_players: int = 5, year: int = 1950, 
               seed: int = 23, starting_pantry: int = 100) -> TurnProcessor:
    """Place kings at balanced starts, seed pantries, return TurnProcessor."""
    self.loadSquads(year=year, seed=seed, levels=n_players)
    starts = self.find_starts(n=n_players, ring_radius=3, min_distance=6)
    
    for squad, start_idx in zip(self.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = self.location_by_index(start_idx)
        move = Move.from_squad(squad, placement=Placement(king, loc))
        move.apply(self)
        king.place.pantry = starting_pantry
    
    self.player_squad = self.squads[0]
    return TurnProcessor(self, self.squads)


@patch
def play_round(self: TurnProcessor, moves: dict[Squad, Move]) -> list[dict]:
    summaries = []
    for squad in self.squads:
        if not squad.king_alive():
            continue
        if squad not in moves:
            continue
        s = self.resolve_move(squad, moves[squad])
        summaries.append(s)
        if self.game_over:
            break
    self.record.snapshot(self.turn, self.parts, 
                         player_squad=self.player_squad, 
                         summaries=summaries)
    self.turn += 1
    return summaries


In [ ]:
#| export
@patch
def play_round(self: TurnProcessor, moves: dict[Squad, Move]) -> list[dict]:
    summaries = []
    self._in_round = True  # ← guard
    for squad in self.squads:
        if not squad.king_alive(): continue
        if squad not in moves: continue
        s = self.resolve_move(squad, moves[squad])
        summaries.append(s)
        if self.game_over: break
    self._in_round = False
    self.summaries_by_turn[self.turn] = summaries
    self.record.snapshot(self.turn, self.parts, 
                         player_squad=self.player_squad, summaries=summaries)
    self.turn += 1
    return summaries




# Reset and replay
myStuff = GameParts()
playerCount = 3
myStuff.grid.adjustRadius(20)
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)

for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    move = Move.from_squad(squad, placement=Placement(king, loc))
    move.apply(myStuff)
    king.place.pantry = 100

tp = TurnProcessor(myStuff, myStuff.squads)

for round_num in range(3):
    print(f"\n=== Purchase round {round_num + 1} ===")
    for squad in myStuff.squads:
        move = squad.best_purchase(myStuff)
        if move:
            summary = tp.resolve_move(squad, move)
            pt = move.purchase.piece_type.value if move.purchase else "?"
            idx = myStuff.hp2i(move.purchase.location.location) if move.purchase else "?"
            print(f"  {squad.name}: bought {pt} → hex {idx}, "
                  f"king_pantry={summary['king_pantry_after']}, "
                  f"connected={summary['connected']}")
        else:
            print(f"  {squad.name}: no purchase (broke or no valid hex)")

print(f"\n{tp.status()}")

!cat ../../HexMagic/magic/pieces.py

!cat ../../HexMagic/magic/moves.py

In [ ]:
#| export
CHART_COLS = [
    ('king_pantry',    '🧺 Pantry'),
    ('total_harvest',  '🌾 Harvest'),
    ('total_diet',     '🍽 Diet'),
    ('net_food',       '📊 Net'),
    ('drain_dealt',    '⚔️ Drain Out'),
    ('drain_taken',    '🛡 Drain In'),
    ('pieces_placed',  '♟ Placed'),
    ('bottlenecks',    '⚠️ Bottleneck'),
    ('sight_coverage', '👁 Sight'),
    ('king_connected', '🔗 Connected'),
    ('isolated',       '🏝 Isolated'),
]

@dataclass
class TurnCharts:
    """Reusable chart panel driven by a TurnProcessor's GameRecord."""
    tp: TurnProcessor
    active_cols: set = field(default_factory=lambda: {'king_pantry', 'pieces_placed'})
    active_squads: set = field(default_factory=lambda: set())   # empty = all
    player_squad: Squad | None = None
    
    # ── HX targets (override if your page layout differs) ──
    hx_target: str = "#chart-panel"
    hx_col_post: str = "/toggle_chart_col"
    hx_squad_post: str = "/toggle_chart_squad"

    @property
    def squads(self) -> list[Squad]: return self.tp.squads
    @property
    def df(self) -> pd.DataFrame: return self.tp.record.df

    # ── Toggle helpers (call from your route handlers) ──
    def toggle_col(self, col: str):
        self.active_cols ^= {col}
        if not self.active_cols:
            self.active_cols = {'king_pantry'}

    def toggle_squad(self, name: str):
        self.active_squads ^= {name}

    # ── Controls ──
    def controls(self):
        col_toggles = DivLAligned(
            *[Button(label,
                cls=(ButtonT.primary if key in self.active_cols else ButtonT.ghost, "text-xs"),
                hx_post=self.hx_col_post,
                hx_vals=json.dumps({"col": key}),
                hx_target=self.hx_target)
              for key, label in CHART_COLS],
            cls="gap-1 flex-wrap",
        )

        active = self.active_squads or {s.name for s in self.squads}
        squad_toggles = []
        for sq in self.squads:
            is_on = sq.name in active or not self.active_squads
            is_player = sq is self.player_squad
            label = DivLAligned(
                Span("●", style=f"color:{sq.flag.primary}; font-size:1.1em"),
                Small(sq.name.split()[0],
                      cls="font-medium text-xs" if is_player else "text-xs"),
            )
            squad_toggles.append(
                Button(label,
                    cls=(ButtonT.primary if is_on else ButtonT.ghost, "text-xs px-2"),
                    hx_post=self.hx_squad_post,
                    hx_vals=json.dumps({"squad": sq.name}),
                    hx_target=self.hx_target))

        return Div(
            DivLAligned(*squad_toggles, cls="gap-1 flex-wrap"),
            col_toggles,
            cls="space-y-1 pb-2 border-b",
        )

    # ── Main render ──
    def __ft__(self):
        if self.df.empty:
            return Div(
                self.controls(),
                P("No history yet — complete a turn.", cls=TextPresets.muted_sm),
                id="chart-panel", cls="space-y-3",
            )

        df = self.df
        active_squads = self.active_squads or {s.name for s in self.squads}
        colors = {sq.name: sq.flag.primary for sq in self.squads}

        cols = sorted(self.active_cols)
        n = len(cols)
        if n == 0:
            return Div(self.controls(), id="chart-panel")
        ncols = min(3, n)
        nrows = math.ceil(n / ncols)

        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(4.5 * ncols, 3.2 * nrows),
                                  facecolor='none', squeeze=False)
        fig.patch.set_alpha(0)
        col_labels = dict(CHART_COLS)

        for i, col in enumerate(cols):
            ax = axes[i // ncols][i % ncols]
            for sq in self.squads:
                if sq.name not in active_squads: continue
                sub = df[df.squad == sq.name].sort_values('turn')
                if sub.empty or col not in sub.columns: continue
                is_player = sq is self.player_squad
                is_alive = sub.iloc[-1]['alive']
                color = colors[sq.name]
                ax.plot(sub['turn'], sub[col],
                        color=color,
                        lw=2.5 if is_player else 1.2,
                        ls='-' if is_alive else '--',
                        alpha=1.0 if is_player else (0.55 if is_alive else 0.2),
                        marker='o', markersize=3 if is_player else 1.5,
                        label=sq.name.split()[0])
                if is_player and not sub.empty:
                    last = sub.iloc[-1]
                    ax.annotate(f"{last[col]:.0f}",
                                xy=(last['turn'], last[col]),
                                fontsize=7, color=color,
                                xytext=(4, 0), textcoords='offset points')
            ax.set_title(col_labels.get(col, col), fontsize=10, fontweight='bold')
            ax.tick_params(labelsize=7)
            ax.grid(alpha=0.2, lw=0.5)
            ax.spines[['top', 'right']].set_visible(False)
            ax.set_xlabel("Turn", fontsize=7)

        for j in range(n, nrows * ncols):
            axes[j // ncols][j % ncols].set_visible(False)

        handles, labels = axes[0][0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc='lower center',
                       ncol=len(active_squads), fontsize=8,
                       framealpha=0.3, bbox_to_anchor=(0.5, -0.02))
        plt.tight_layout(rect=[0, 0.06, 1, 1])

        buf = io.BytesIO()
        fig.savefig(buf, format='svg', bbox_inches='tight', transparent=True)
        plt.close(fig)
        buf.seek(0)

        return Div(
            self.controls(),
            NotStr(buf.read().decode()),
            id="chart-panel",
            cls="space-y-2",
        )


We have some code that shows how the game progresses over turns and I think we could make this into some sort of a class helper with our turn processor.
```python
@patch
def _chart_controls(GS:GlobalGameState):
    """Two toggle rows: columns and squads."""
    # Row 1: metric columns
    col_toggles = DivLAligned(
        *[Button(label,
            cls=(ButtonT.primary if key in GS.active_chart_cols else ButtonT.ghost, "text-xs"),
            hx_post="/toggle_chart_col",
            hx_vals=json.dumps({"col": key}),
            hx_target="#chart-panel")
          for key, label in CHART_COLS],
        cls="gap-1 flex-wrap",
    )

    # Row 2: squads (flag color dots + name)
    squad_toggles = []
    all_squads = GS.parts.squads if GS.parts else []
    active = GS.active_chart_squads or {s.name for s in all_squads}  # empty = all shown

    for sq in all_squads:
        is_on = sq.name in active or not GS.active_chart_squads
        is_player = sq is GS.player_squad
        label = DivLAligned(
            Span("●", style=f"color:{sq.flag.primary}; font-size:1.1em"),
            Small(sq.name.split()[0], cls="font-medium text-xs" if is_player else "text-xs"),
        )
        squad_toggles.append(
            Button(label,
                cls=(ButtonT.primary if is_on else ButtonT.ghost, "text-xs px-2"),
                hx_post="/toggle_chart_squad",
                hx_vals=json.dumps({"squad": sq.name}),
                hx_target="#chart-panel")
        )

    return Div(
        DivLAligned(*squad_toggles, cls="gap-1 flex-wrap"),
        col_toggles,
        cls="space-y-1 pb-2 border-b",
    )

@patch
def panel_charts(GS:GlobalGameState):
    """Time-series sparklines — controlled by active_chart_cols + active_chart_squads."""
    if GS.tp is None or GS.tp.record.df.empty:
        return Div(
            _chart_controls(),
            P("No history yet — complete a turn.", cls=TextPresets.muted_sm),
            id="chart-panel", cls="space-y-3",
        )

    df = GS.tp.record.df
    all_squads = GS.parts.squads
    
    # Which squads to plot
    active_squads = GS.active_chart_squads or {s.name for s in all_squads}
    colors = {sq.name: sq.flag.primary for sq in all_squads}
    
    cols = sorted(GS.active_chart_cols)
    n = len(cols)
    ncols = min(3, n)
    nrows = math.ceil(n / ncols)
    
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(4.5 * ncols, 3.2 * nrows),
                              facecolor='none', squeeze=False)
    fig.patch.set_alpha(0)

    col_labels = dict(CHART_COLS)

    for i, col in enumerate(cols):
        ax = axes[i // ncols][i % ncols]
        
        for sq in all_squads:
            if sq.name not in active_squads:
                continue
            sub = df[df.squad == sq.name].sort_values('turn')
            if sub.empty or col not in sub.columns:
                continue
            is_player = sq is GS.player_squad
            is_alive = sub.iloc[-1]['alive']
            color = colors[sq.name]
            ax.plot(sub['turn'], sub[col],
                    color=color,
                    lw=2.5 if is_player else 1.2,
                    ls='-' if is_alive else '--',
                    alpha=1.0 if is_player else (0.55 if is_alive else 0.2),
                    marker='o', markersize=3 if is_player else 1.5,
                    label=sq.name.split()[0])
            if is_player and not sub.empty:
                last = sub.iloc[-1]
                ax.annotate(f"{last[col]:.0f}",
                            xy=(last['turn'], last[col]),
                            fontsize=7, color=color,
                            xytext=(4, 0), textcoords='offset points')

        ax.set_title(col_labels.get(col, col), fontsize=10, fontweight='bold')
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.2, lw=0.5)
        ax.spines[['top', 'right']].set_visible(False)
        ax.set_xlabel("Turn", fontsize=7)

    # Hide unused axes
    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].set_visible(False)

    # Shared legend
    handles, labels = axes[0][0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='lower center',
                   ncol=len(active_squads), fontsize=8,
                   framealpha=0.3, bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout(rect=[0, 0.06, 1, 1])

    buf = io.BytesIO()
    fig.savefig(buf, format='svg', bbox_inches='tight', transparent=True)
    plt.close(fig)
    buf.seek(0)

    return Div(
        GS._chart_controls(),
        NotStr(buf.read().decode()),
        id="chart-panel",
        cls="space-y-2",
    )
```

for reference this is GlobalGameState
```python
@dataclass
class GlobalGameState:
    parts: GameParts | None = None
    tp: TurnProcessor | None = None
    player_squad: Squad | None = None
    selected_piece: Piece | None = None
    pending_rotations: list[Rotation] = field(default_factory=list)
    pending_placement: Placement | None = None
    pending_purchase: Purchase | None = None
    pending_purchase_type: PieceType | None = None  # add this
    active_overlays: set = field(default_factory=lambda: { 'rivers','lanterns','network','ocean'})
    turn: int = 0
    radius:float = 25
    debug_overlays: bool = False
    rotationClickId: int = None
    ai_strategies: dict[str, PlacementStrategy] = field(default_factory=dict)
    focused_piece: Piece | None = None
    # Add to GlobalGameState
    active_chart_cols: set = field(default_factory=lambda: {'king_pantry', 'pieces_placed'})
    active_chart_squads: set = field(default_factory=lambda: set())  # empty = all
    
    def select_piece(self, piece: Piece):
        """Select an unplaced piece for placement."""
        if piece.place is not None:
            return  # already on board — rotation handled separately
        self.selected_piece = piece
    
    def place_piece(self, hex_idx: int):
        """Place the selected piece at a hex."""
        if self.selected_piece is None:
            return False
        loc = self.parts.location_by_index(hex_idx)
        self.pending_placement = Placement(self.selected_piece, loc)
        self.selected_piece.place = loc  # preview immediately
        self.selected_piece = None
        return True
    
    def unplace_piece(self, piece: Piece):
        """Remove a pending placement (before committing)."""
        if self.pending_placement and self.pending_placement.piece is piece:
            piece.place = None
            self.pending_placement = None
    
    
    @property
    def is_active(self) -> bool:
        return self.parts is not None and self.tp is not None
    
    @property
    def game_over(self) -> bool:
        return self.tp.game_over if self.tp else False

```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at this, the charting logic just needs access to `tp.record.df`, the squad list, and some UI toggle state. We can pull it into a standalone helper that pairs with `TurnProcessor`:

```python
CHART_COLS = [
    ('king_pantry',    '🧺 Pantry'),
    ('total_harvest',  '🌾 Harvest'),
    ('total_diet',     '🍽 Diet'),
    ('net_food',       '📊 Net'),
    ('drain_dealt',    '⚔️ Drain Out'),
    ('drain_taken',    '🛡 Drain In'),
    ('pieces_placed',  '♟ Placed'),
    ('bottlenecks',    '⚠️ Bottleneck'),
    ('sight_coverage', '👁 Sight'),
    ('king_connected', '🔗 Connected'),
    ('isolated',       '🏝 Isolated'),
]

@dataclass
class TurnCharts:
    """Reusable chart panel driven by a TurnProcessor's GameRecord."""
    tp: TurnProcessor
    active_cols: set = field(default_factory=lambda: {'king_pantry', 'pieces_placed'})
    active_squads: set = field(default_factory=lambda: set())   # empty = all
    player_squad: Squad | None = None
    
    # ── HX targets (override if your page layout differs) ──
    hx_target: str = "#chart-panel"
    hx_col_post: str = "/toggle_chart_col"
    hx_squad_post: str = "/toggle_chart_squad"

    @property
    def squads(self) -> list[Squad]: return self.tp.squads
    @property
    def df(self) -> pd.DataFrame: return self.tp.record.df

    # ── Toggle helpers (call from your route handlers) ──
    def toggle_col(self, col: str):
        self.active_cols ^= {col}
        if not self.active_cols:
            self.active_cols = {'king_pantry'}

    def toggle_squad(self, name: str):
        self.active_squads ^= {name}

    # ── Controls ──
    def controls(self):
        col_toggles = DivLAligned(
            *[Button(label,
                cls=(ButtonT.primary if key in self.active_cols else ButtonT.ghost, "text-xs"),
                hx_post=self.hx_col_post,
                hx_vals=json.dumps({"col": key}),
                hx_target=self.hx_target)
              for key, label in CHART_COLS],
            cls="gap-1 flex-wrap",
        )

        active = self.active_squads or {s.name for s in self.squads}
        squad_toggles = []
        for sq in self.squads:
            is_on = sq.name in active or not self.active_squads
            is_player = sq is self.player_squad
            label = DivLAligned(
                Span("●", style=f"color:{sq.flag.primary}; font-size:1.1em"),
                Small(sq.name.split()[0],
                      cls="font-medium text-xs" if is_player else "text-xs"),
            )
            squad_toggles.append(
                Button(label,
                    cls=(ButtonT.primary if is_on else ButtonT.ghost, "text-xs px-2"),
                    hx_post=self.hx_squad_post,
                    hx_vals=json.dumps({"squad": sq.name}),
                    hx_target=self.hx_target))

        return Div(
            DivLAligned(*squad_toggles, cls="gap-1 flex-wrap"),
            col_toggles,
            cls="space-y-1 pb-2 border-b",
        )

    # ── Main render ──
    def __ft__(self):
        if self.df.empty:
            return Div(
                self.controls(),
                P("No history yet — complete a turn.", cls=TextPresets.muted_sm),
                id="chart-panel", cls="space-y-3",
            )

        df = self.df
        active_squads = self.active_squads or {s.name for s in self.squads}
        colors = {sq.name: sq.flag.primary for sq in self.squads}

        cols = sorted(self.active_cols)
        n = len(cols)
        if n == 0:
            return Div(self.controls(), id="chart-panel")
        ncols = min(3, n)
        nrows = math.ceil(n / ncols)

        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(4.5 * ncols, 3.2 * nrows),
                                  facecolor='none', squeeze=False)
        fig.patch.set_alpha(0)
        col_labels = dict(CHART_COLS)

        for i, col in enumerate(cols):
            ax = axes[i // ncols][i % ncols]
            for sq in self.squads:
                if sq.name not in active_squads: continue
                sub = df[df.squad == sq.name].sort_values('turn')
                if sub.empty or col not in sub.columns: continue
                is_player = sq is self.player_squad
                is_alive = sub.iloc[-1]['alive']
                color = colors[sq.name]
                ax.plot(sub['turn'], sub[col],
                        color=color,
                        lw=2.5 if is_player else 1.2,
                        ls='-' if is_alive else '--',
                        alpha=1.0 if is_player else (0.55 if is_alive else 0.2),
                        marker='o', markersize=3 if is_player else 1.5,
                        label=sq.name.split()[0])
                if is_player and not sub.empty:
                    last = sub.iloc[-1]
                    ax.annotate(f"{last[col]:.0f}",
                                xy=(last['turn'], last[col]),
                                fontsize=7, color=color,
                                xytext=(4, 0), textcoords='offset points')
            ax.set_title(col_labels.get(col, col), fontsize=10, fontweight='bold')
            ax.tick_params(labelsize=7)
            ax.grid(alpha=0.2, lw=0.5)
            ax.spines[['top', 'right']].set_visible(False)
            ax.set_xlabel("Turn", fontsize=7)

        for j in range(n, nrows * ncols):
            axes[j // ncols][j % ncols].set_visible(False)

        handles, labels = axes[0][0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc='lower center',
                       ncol=len(active_squads), fontsize=8,
                       framealpha=0.3, bbox_to_anchor=(0.5, -0.02))
        plt.tight_layout(rect=[0, 0.06, 1, 1])

        buf = io.BytesIO()
        fig.savefig(buf, format='svg', bbox_inches='tight', transparent=True)
        plt.close(fig)
        buf.seek(0)

        return Div(
            self.controls(),
            NotStr(buf.read().decode()),
            id="chart-panel",
            cls="space-y-2",
        )
```

Key changes from the `GlobalGameState` version:

- **Self-contained** — `TurnCharts` owns its own toggle state (`active_cols`, `active_squads`) and gets everything else from the `TurnProcessor` reference.
- **`toggle_col` / `toggle_squad`** — simple methods your route handlers call instead of reaching into GS fields.
- **`__ft__`** — renders the full panel, so you can just drop a `TurnCharts` instance into any FT tree.
- **Configurable HX targets** — `hx_target`, `hx_col_post`, `hx_squad_post` are constructor args, so it works in different page layouts without editing the class.

Then in `GlobalGameState` you'd replace the chart fields + methods with one field:

```python
@dataclass
class GlobalGameState:
    # ... existing fields ...
    charts: TurnCharts | None = None
    
    # after setting up tp:
    # self.charts = TurnCharts(tp=self.tp, player_squad=self.player_squad)
```

And your routes become:

```python
@rt("/toggle_chart_col")
def post(col: str):
    GS.charts.toggle_col(col)
    return GS.charts

@rt("/toggle_chart_squad")
def post(squad: str):
    GS.charts.toggle_squad(squad)
    return GS.charts
```

Don't forget `import json` at the top of the module — the controls use `json.dumps` for `hx_vals`.

In [ ]:
#| export
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, Objective, Constraint,
    Binary, maximize, SolverFactory, value
)

In [ ]:
#| export
@patch
def optimal_pawn_placements(squad: Squad, parts: GameParts, k: int = 3) -> list[int]:
    """
    Warehouse-location MIP: choose k hexes from ally sight to maximise
    total harvest tier. Returns list of up to k hex indices.
    """
    placed = [p for p in squad.pieces if p.place is not None]
    if not placed:
        return []

    terr  = parts.terr
    tiers = parts.fy.tiers
    occupied = {
        parts.hp2i(p.place.location)
        for sq in parts.squads
        for p in sq.pieces if p.place is not None
    }

    # Union of sight across all facings for all placed allies
    dirs = HexPosition.directions()
    visible = set()
    for ally in placed:
        bonus = ally.food.bonus_sight
        eff   = max(2, ally.place.level) + bonus
        for facing in dirs:
            visible |= {
                parts.hp2i(hp)
                for hp in ally.place.location.field_of_view(facing, eff)
            }

    # Filter to valid land, unoccupied hexes
    candidates = sorted(
        idx for idx in visible
        if 0 <= idx < len(terr.elevations)
        and terr.elevations[idx] > 0
        and idx not in occupied
    )
    if not candidates:
        return []

    yield_map = {idx: float(tiers[idx]) for idx in candidates}

    # ── Pyomo model ──
    m = ConcreteModel()
    m.C = Set(initialize=candidates)
    m.y = {idx: yield_map[idx] for idx in candidates}
    m.x = Var(m.C, domain=Binary)

    m.obj = Objective(
        expr=sum(m.y[i] * m.x[i] for i in m.C),
        sense=maximize,
    )
    m.budget = Constraint(
        expr=sum(m.x[i] for i in m.C) <= k
    )

    solver = SolverFactory('appsi_highs')
    result = solver.solve(m, tee=False)

    chosen = [i for i in candidates if value(m.x[i]) > 0.5]
    return chosen[:k]


In [ ]:
#| export
@patch
def place_optimal_pawns(self: Squad, parts: GameParts, 
                        tp: TurnProcessor, k: int = 3) -> list[dict]:
    """
    Solve for k optimal pawn hexes, then build a valid Move for each
    (including rotation if needed to bring the hex into sight).
    """
    summaries = []
    dirs = HexPosition.directions()
    
    for _ in range(k):
        chosen = self.optimal_pawn_placements(parts, k=1)
        if not chosen:
            break
        target_idx = chosen[0]
        
        pawn = next(
            (p for p in self.pieces 
             if p.type == PieceType.PAWN and p.place is None), None
        )
        if pawn is None:
            break
        
        # Check if target is already in current sight
        placed = [p for p in self.pieces if p.place is not None]
        current_visible = set()
        for ally in placed:
            current_visible |= ally.sight().hexes
        
        rotation = None
        if target_idx not in current_visible:
            # Find which single rotation brings it into sight
            best_rot = None
            best_extra = -1  # prefer the rotation that also reveals the most new hexes
            
            for ally in placed:
                bonus = ally.food.bonus_sight
                eff = max(2, ally.place.level) + bonus
                for new_facing in dirs:
                    if new_facing == ally.place.facing:
                        continue
                    new_sight = set(
                        parts.hp2i(hp)
                        for hp in ally.place.location.field_of_view(new_facing, eff)
                    )
                    if target_idx in new_sight:
                        # Score: how many extra useful hexes does this rotation give?
                        extra = len(new_sight - current_visible)
                        if extra > best_extra:
                            best_extra = extra
                            best_rot = Rotation(ally, new_facing)
            
            if best_rot is None:
                # Can't reach this hex with a single rotation — skip
                continue
            rotation = best_rot
        
        loc = parts.location_by_index(target_idx)
        rotations = [rotation] if rotation else []
        
        try:
            move = Move.from_squad(self, rotations=rotations,
                                   placement=Placement(pawn, loc))
            summary = tp.resolve_move(self, move)
            summaries.append(summary)
        except MoveError as e:
            print(f"  ⚠️ {self.name}: skipped hex {target_idx} — {e}")
            continue
    
    return summaries


In [ ]:
#| export
Squad.__hash__ = lambda self: id(self)


# ── Fresh game ──
myStuff = GameParts()
myStuff.grid.adjustRadius(20)
playerCount = 3
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)

# Reset any stale placement state
for squad in myStuff.squads:
    for p in squad.pieces:
        p.place = None

starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)

for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc  = myStuff.location_by_index(start_idx)
    Move.from_squad(squad, placement=Placement(king, loc)).apply(myStuff)
    king.place.pantry = 100

tp = TurnProcessor(myStuff, myStuff.squads)
# ── Optimal pawn build-out ──
print("=== Optimal pawn placement ===")
for squad in myStuff.squads:
    summaries = squad.place_optimal_pawns(myStuff, tp, k=3)
    for s in summaries:
        print(f"  {squad.name}: placed pawn, king_pantry={s['king_pantry_after']}, "
              f"connected={s['connected']}")

print(f"\n{tp.status()}")

# ── A few combat/food rounds ──
print("\n=== Simulation rounds ===")
for round_num in range(5):
    moves = {sq: Move() for sq in myStuff.squads if sq.king_alive()}
    summaries = tp.play_round(moves)
    for s in summaries:
        print(f"  T{tp.turn-1} {s['squad']}: "
              f"harvested={s['food_harvested']:.0f}, diet={s['food_diet']:.0f}, "
              f"drained={s['food_drain_dealt']:.0f}, starved={s['starved']}, "
              f"king_pantry={s['king_pantry_after']}")
    if tp.game_over:
        print(f"\n🏆 Winner: {tp.winner.name if tp.winner else 'nobody'}")
        break

print(f"\n{tp.status()}")


In [ ]:
#| export
@dataclass
class TurnCell:
    squad_name: str
    turn: int
    pantry_before: int
    pantry_after: int
    harvest: int
    diet: int
    drain_edges: list[DrainEdge]     # who we attacked
    drain_taken: int                  # how much we lost
    placed: list[str]                # piece names added this turn
    starved: list[str]
    eliminated: str | None

    @classmethod
    def from_summary(cls: TurnCell, summary: dict, squad: Squad) -> 'TurnCell':
        drain_edges = [DrainEdge(*e) for e in summary.get('drain_edges', [])]
        return cls(
            squad_name=summary['squad'],
            turn=summary['turn'],
            pantry_before=summary['king_pantry_before'],
            pantry_after=summary['king_pantry_after'],
            harvest=int(summary['food_harvested']),
            diet=int(summary['food_diet']),
            drain_edges=drain_edges,
            drain_taken=summary.get('drain_taken', 0),
            placed=[summary['placed_name']] if summary.get('placed_name') else [],
            starved=summary.get('starved', []),
            eliminated=summary.get('eliminated'),
        )


In [ ]:
#| export

@patch
def __ft__(self: TurnCell, squad: Squad) -> FT:
    flag = squad.flag
    delta = self.pantry_after - self.pantry_before
    delta_color = "#27ae60" if delta >= 0 else "#e74c3c"
    delta_str   = f"+{delta}" if delta >= 0 else str(delta)

    # Pantry row
    pantry_row = DivFullySpaced(
        Span(f"🧺 {self.pantry_before} → {self.pantry_after}",
             style="font-size:12px;font-weight:700;"),
        Span(delta_str,
             style=f"font-size:12px;font-weight:800;color:{delta_color};"),
    )

    # Food row
    food_row = DivLAligned(
        Span(f"🌾 +{self.harvest}", style="font-size:11px;color:#27ae60;font-weight:600;"),
        Span(f"🍖 -{self.diet}",    style="font-size:11px;color:#e67e22;font-weight:600;"),
        style="gap:10px;"
    )

    # Placed pieces
    placed_chips = [
        Span(f"📍 {name}",
             style=f"font-size:10px;background:{flag.lightComp};"
                   f"color:{flag.darkPrimary};border-radius:5px;padding:1px 6px;")
        for name in self.placed
    ]

    # Drain dealt
    attack_chips = [
        Span(f"⚔️ →{e.target_squad}: {e.amount}",
             style="font-size:10px;background:#fde8e8;color:#c0392b;"
                   "border-radius:5px;padding:1px 6px;")
        for e in self.drain_edges
    ]

    # Starved
    starved_chips = [
        Span(f"💀 {name}",
             style="font-size:10px;background:#f5f5f5;color:#888;"
                   "border-radius:5px;padding:1px 6px;")
        for name in self.starved
    ]

    # Eliminated banner
    elim = []
    if self.eliminated:
        elim = [Div(f"☠️ {self.eliminated} eliminated",
                    style="font-size:11px;color:white;background:#c0392b;"
                          "border-radius:6px;padding:2px 8px;text-align:center;margin-top:4px;")]

    chips_row = Div(
        *placed_chips, *attack_chips, *starved_chips,
        style="display:flex;flex-wrap:wrap;gap:4px;margin-top:4px;"
    )

    return Div(
        pantry_row, food_row, chips_row, *elim,
        style=f"padding:8px;border-radius:10px;background:white;"
              f"border:1px solid {flag.lightComp};min-width:140px;"
    )

@patch
def chips(self: TurnCell, squad: Squad) -> FT:
    """Ultra-compact badge for list view."""
    flag  = squad.flag
    delta = self.pantry_after - self.pantry_before
    color = "#27ae60" if delta >= 0 else "#e74c3c"
    d_str = f"+{delta}" if delta >= 0 else str(delta)

    parts = [
        NotStr(flag.logo(size=20, piece_id=f"tc_{self.squad_name}_{self.turn}")),
        Span(self.squad_name, style="font-size:11px;font-weight:700;"),
        Span(f"🧺{d_str}", style=f"font-size:11px;color:{color};font-weight:700;"),
    ]
    if self.placed:
        parts.append(Span(f"📍×{len(self.placed)}", style="font-size:10px;"))
    if self.drain_edges:
        total_drain = sum(e.amount for e in self.drain_edges)
        parts.append(Span(f"⚔️{total_drain}", style="font-size:10px;color:#c0392b;"))
    if self.starved:
        parts.append(Span(f"💀×{len(self.starved)}", style="font-size:10px;color:#888;"))
    if self.eliminated:
        parts.append(Span("☠️", style="font-size:12px;"))

    return Div(*parts,
               style=f"display:flex;align-items:center;gap:6px;padding:4px 8px;"
                     f"border-radius:8px;background:{flag.lightComp}20;"
                     f"border:1px solid {flag.lightComp};")


I think I added tracked pieces. anything else I need to add?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at the summary dict vs TurnCell fields, I see two gaps:

1. **`drain_taken`** — the summary tracks `food_drain_dealt` (what *you* drained from others), but not how much *others* drained from *you*. You'd need to cross-reference other squads' `drain_edges` to compute this. So `from_summary` would need *all* summaries from the round, not just one.

2. **`expired`** — pieces removed by lifespan timeout are already in the summary (`summary['expired']`), but TurnCell doesn't have a field for them. Worth adding?

Lets comeback to those. Can you write our tables

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
@patch
@classmethod
def from_summary(cls: TurnCell, summary: dict, squad: Squad) -> 'TurnCell':
    drain_edges = [DrainEdge(*e) for e in summary.get('drain_edges', [])]
    return cls(
        squad_name=summary['squad'],
        turn=summary['turn'],
        pantry_before=summary['king_pantry_before'],
        pantry_after=summary['king_pantry_after'],
        harvest=int(summary['food_harvested']),
        diet=int(summary['food_diet']),
        drain_edges=drain_edges,
        drain_taken=summary.get('drain_taken', 0),
        placed=[summary['placed_name']] if summary.get('placed_name') else [],
        starved=summary.get('starved', []),
        eliminated=summary.get('eliminated'),
    )

@patch
def __ft__(self: TurnCell, squad: Squad) -> FT:
    flag = squad.flag
    delta = self.pantry_after - self.pantry_before
    delta_color = "#27ae60" if delta >= 0 else "#e74c3c"
    delta_str   = f"+{delta}" if delta >= 0 else str(delta)

    # Pantry row
    pantry_row = DivFullySpaced(
        Span(f"🧺 {self.pantry_before} → {self.pantry_after}",
             style="font-size:12px;font-weight:700;"),
        Span(delta_str,
             style=f"font-size:12px;font-weight:800;color:{delta_color};"),
    )

    # Food row
    food_row = DivLAligned(
        Span(f"🌾 +{self.harvest}", style="font-size:11px;color:#27ae60;font-weight:600;"),
        Span(f"🍖 -{self.diet}",    style="font-size:11px;color:#e67e22;font-weight:600;"),
        style="gap:10px;"
    )

    # Placed pieces
    placed_chips = [
        Span(f"📍 {name}",
             style=f"font-size:10px;background:{flag.lightComp};"
                   f"color:{flag.darkPrimary};border-radius:5px;padding:1px 6px;")
        for name in self.placed
    ]

    # Drain dealt
    attack_chips = [
        Span(f"⚔️ →{e.target_squad}: {e.amount}",
             style="font-size:10px;background:#fde8e8;color:#c0392b;"
                   "border-radius:5px;padding:1px 6px;")
        for e in self.drain_edges
    ]

    # Starved
    starved_chips = [
        Span(f"💀 {name}",
             style="font-size:10px;background:#f5f5f5;color:#888;"
                   "border-radius:5px;padding:1px 6px;")
        for name in self.starved
    ]

    # Eliminated banner
    elim = []
    if self.eliminated:
        elim = [Div(f"☠️ {self.eliminated} eliminated",
                    style="font-size:11px;color:white;background:#c0392b;"
                          "border-radius:6px;padding:2px 8px;text-align:center;margin-top:4px;")]

    chips_row = Div(
        *placed_chips, *attack_chips, *starved_chips,
        style="display:flex;flex-wrap:wrap;gap:4px;margin-top:4px;"
    )

    return Div(
        pantry_row, food_row, chips_row, *elim,
        style=f"padding:8px;border-radius:10px;background:white;"
              f"border:1px solid {flag.lightComp};min-width:140px;"
    )

@patch
def chips(self: TurnCell, squad: Squad) -> FT:
    """Ultra-compact badge for list view."""
    flag  = squad.flag
    delta = self.pantry_after - self.pantry_before
    color = "#27ae60" if delta >= 0 else "#e74c3c"
    d_str = f"+{delta}" if delta >= 0 else str(delta)

    parts = [
        NotStr(flag.logo(size=20, piece_id=f"tc_{self.squad_name}_{self.turn}")),
        Span(self.squad_name, style="font-size:11px;font-weight:700;"),
        Span(f"🧺{d_str}", style=f"font-size:11px;color:{color};font-weight:700;"),
    ]
    if self.placed:
        parts.append(Span(f"📍×{len(self.placed)}", style="font-size:10px;"))
    if self.drain_edges:
        total_drain = sum(e.amount for e in self.drain_edges)
        parts.append(Span(f"⚔️{total_drain}", style="font-size:10px;color:#c0392b;"))
    if self.starved:
        parts.append(Span(f"💀×{len(self.starved)}", style="font-size:10px;color:#888;"))
    if self.eliminated:
        parts.append(Span("☠️", style="font-size:12px;"))

    return Div(*parts,
               style=f"display:flex;align-items:center;gap:6px;padding:4px 8px;"
                     f"border-radius:8px;background:{flag.lightComp}20;"
                     f"border:1px solid {flag.lightComp};")
```

Now the two tables:

```python
def turnTable(tp: TurnProcessor, summaries_by_turn: dict[int, list[dict]]) -> FT:
    """Turn × Squad grid of TurnCell cards."""
    squads   = tp.squads
    sq_map   = {sq.name: sq for sq in squads}
    all_turns = sorted(summaries_by_turn.keys())

    header = Tr(
        Th("Turn", style="padding:8px;font-size:11px;text-align:center;"),
        *[
            Th(
                DivVStacked(
                    NotStr(sq.flag.logo(size=28, piece_id=f"tth_{sq.name}")),
                    Span(sq.name, style=f"font-size:10px;font-weight:700;"
                                        f"color:{sq.flag.darkPrimary};"),
                    style="align-items:center;gap:3px;"
                ),
                style=f"padding:8px;min-width:160px;text-align:center;"
                      f"background:{sq.flag.lightComp};"
                      f"border-bottom:2px solid {sq.flag.primary};"
            )
            for sq in squads
        ]
    )

    body_rows = []
    for turn in all_turns:
        summaries = summaries_by_turn[turn]
        cell_map  = {s['squad']: s for s in summaries}

        cells = [Td(
            Span(f"T{turn}", style="font-size:12px;font-weight:800;font-family:monospace;"),
            style="padding:6px 8px;text-align:center;vertical-align:top;"
                  "border-right:1px solid #eee;"
        )]

        for sq in squads:
            if sq.name not in cell_map:
                cells.append(Td(
                    Div("—", style="color:#ddd;text-align:center;padding:24px 8px;"),
                    style=f"vertical-align:top;background:{sq.flag.lightComp}10;"
                ))
            else:
                tc = TurnCell.from_summary(cell_map[sq.name], sq)
                cells.append(Td(
                    tc.__ft__(sq),
                    style=f"vertical-align:top;padding:4px;"
                          f"background:{sq.flag.lightComp}10;"
                          f"border-left:2px solid {sq.flag.lightComp};"
                ))

        body_rows.append(Tr(*cells, style="border-bottom:1px solid #f0f0f0;"))

    return Div(
        DivFullySpaced(
            H4("Turn History", cls="font-bold text-base"),
            Small(f"{len(all_turns)} turns · {len(squads)} squads",
                  cls=TextPresets.muted_sm),
            cls="mb-2"
        ),
        Div(
            Table(
                Thead(header), Tbody(*body_rows),
                style="border-collapse:collapse;width:100%;"
            ),
            style="overflow-x:auto;border:1px solid #e5e5e5;border-radius:12px;"
        ),
    )


def turnLogDisplay(tp: TurnProcessor, summaries_by_turn: dict[int, list[dict]]) -> FT:
    """Scrollable list — one row per turn, all squads as chips, expandable detail."""
    sq_map    = {sq.name: sq for sq in tp.squads}
    all_turns = sorted(summaries_by_turn.keys(), reverse=True)  # newest first

    rows = []
    for turn in all_turns:
        summaries = summaries_by_turn[turn]
        detail_id = f"tl-detail-{turn}"

        chip_row = Div(
            *[TurnCell.from_summary(s, sq_map[s['squad']]).chips(sq_map[s['squad']])
              for s in summaries if s['squad'] in sq_map],
            style="display:flex;flex-wrap:wrap;gap:6px;flex:1;"
        )

        detail_cells = Div(
            *[TurnCell.from_summary(s, sq_map[s['squad']]).__ft__(sq_map[s['squad']])
              for s in summaries if s['squad'] in sq_map],
            style="display:flex;flex-wrap:wrap;gap:8px;padding:8px 0;"
        )

        rows.append(Div(
            Div(
                Div(f"T{turn}",
                    style="font-size:11px;font-weight:800;font-family:monospace;"
                          "background:#f5f5f5;border-radius:6px;padding:3px 8px;"
                          "flex-shrink:0;"),
                chip_row,
                Button("▾", style="font-size:16px;background:none;border:none;cursor:pointer;",
                       onclick=f"document.getElementById('{detail_id}').classList.toggle('hidden')"),
                style="display:flex;align-items:center;gap:8px;padding:8px 4px;"
                      "border-bottom:1px solid #f0f0f0;"
            ),
            Div(detail_cells, id=detail_id, cls="hidden"),
        ))

    return Div(
        DivFullySpaced(
            H4("Turn Log", cls="font-bold text-base"),
            Small(f"{len(all_turns)} turns", cls=TextPresets.muted_sm),
        ),
        Div(*rows,
            style="max-height:520px;overflow-y:auto;border:1px solid #e5e5e5;"
                  "border-radius:12px;padding:4px 8px;"),
        id="turn-log",
    )
```

Note that both functions take a `summaries_by_turn: dict[int, list[dict]]` — you'd populate that in `play_round` by collecting the summaries list per turn, e.g. `summaries_by_turn[tp.turn] = summaries`. Want me to show how to wire that into `TurnProcessor`?

In [ ]:
#| export
@patch
def turnTable(tp: TurnProcessor, summaries_by_turn: dict[int, list[dict]]) -> FT:
    """Turn × Squad grid of TurnCell cards."""
    squads   = tp.squads
    sq_map   = {sq.name: sq for sq in squads}
    all_turns = sorted(summaries_by_turn.keys())

    header = Tr(
        Th("Turn", style="padding:8px;font-size:11px;text-align:center;"),
        *[
            Th(
                DivVStacked(
                    NotStr(sq.flag.logo(size=28, piece_id=f"tth_{sq.name}")),
                    Span(sq.name, style=f"font-size:10px;font-weight:700;"
                                        f"color:{sq.flag.darkPrimary};"),
                    style="align-items:center;gap:3px;"
                ),
                style=f"padding:8px;min-width:160px;text-align:center;"
                      f"background:{sq.flag.lightComp};"
                      f"border-bottom:2px solid {sq.flag.primary};"
            )
            for sq in squads
        ]
    )

    body_rows = []
    for turn in all_turns:
        summaries = summaries_by_turn[turn]
        cell_map  = {s['squad']: s for s in summaries}

        cells = [Td(
            Span(f"T{turn}", style="font-size:12px;font-weight:800;font-family:monospace;"),
            style="padding:6px 8px;text-align:center;vertical-align:top;"
                  "border-right:1px solid #eee;"
        )]

        for sq in squads:
            if sq.name not in cell_map:
                cells.append(Td(
                    Div("—", style="color:#ddd;text-align:center;padding:24px 8px;"),
                    style=f"vertical-align:top;background:{sq.flag.lightComp}10;"
                ))
            else:
                tc = TurnCell.from_summary(cell_map[sq.name], sq)
                cells.append(Td(
                    tc.__ft__(sq),
                    style=f"vertical-align:top;padding:4px;"
                          f"background:{sq.flag.lightComp}10;"
                          f"border-left:2px solid {sq.flag.lightComp};"
                ))

        body_rows.append(Tr(*cells, style="border-bottom:1px solid #f0f0f0;"))

    return Div(
        DivFullySpaced(
            H4("Turn History", cls="font-bold text-base"),
            Small(f"{len(all_turns)} turns · {len(squads)} squads",
                  cls=TextPresets.muted_sm),
            cls="mb-2"
        ),
        Div(
            Table(
                Thead(header), Tbody(*body_rows),
                style="border-collapse:collapse;width:100%;"
            ),
            style="overflow-x:auto;border:1px solid #e5e5e5;border-radius:12px;"
        ),
    )

@patch
def turnLogDisplay(tp: TurnProcessor, summaries_by_turn: dict[int, list[dict]]) -> FT:
    """Scrollable list — one row per turn, all squads as chips, expandable detail."""
    sq_map    = {sq.name: sq for sq in tp.squads}
    all_turns = sorted(summaries_by_turn.keys(), reverse=True)  # newest first

    rows = []
    for turn in all_turns:
        summaries = summaries_by_turn[turn]
        detail_id = f"tl-detail-{turn}"

        chip_row = Div(
            *[TurnCell.from_summary(s, sq_map[s['squad']]).chips(sq_map[s['squad']])
              for s in summaries if s['squad'] in sq_map],
            style="display:flex;flex-wrap:wrap;gap:6px;flex:1;"
        )

        detail_cells = Div(
            *[TurnCell.from_summary(s, sq_map[s['squad']]).__ft__(sq_map[s['squad']])
              for s in summaries if s['squad'] in sq_map],
            style="display:flex;flex-wrap:wrap;gap:8px;padding:8px 0;"
        )

        rows.append(Div(
            Div(
                Div(f"T{turn}",
                    style="font-size:11px;font-weight:800;font-family:monospace;"
                          "background:#f5f5f5;border-radius:6px;padding:3px 8px;"
                          "flex-shrink:0;"),
                chip_row,
                Button("▾", style="font-size:16px;background:none;border:none;cursor:pointer;",
                       onclick=f"document.getElementById('{detail_id}').classList.toggle('hidden')"),
                style="display:flex;align-items:center;gap:8px;padding:8px 4px;"
                      "border-bottom:1px solid #f0f0f0;"
            ),
            Div(detail_cells, id=detail_id, cls="hidden"),
        ))

    return Div(
        DivFullySpaced(
            H4("Turn Log", cls="font-bold text-base"),
            Small(f"{len(all_turns)} turns", cls=TextPresets.muted_sm),
        ),
        Div(*rows,
            style="max-height:520px;overflow-y:auto;border:1px solid #e5e5e5;"
                  "border-radius:12px;padding:4px 8px;"),
        id="turn-log",
    )


turnTable(tp, tp.summaries_by_turn)

turnLogDisplay(tp, tp.summaries_by_turn)

In [ ]:
# ── Fresh game ──
myStuff = GameParts()
myStuff.grid.adjustRadius(20)
playerCount = 3
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)

# Reset any stale placement state
for squad in myStuff.squads:
    for p in squad.pieces:
        p.place = None

starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)

for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc  = myStuff.location_by_index(start_idx)
    Move.from_squad(squad, placement=Placement(king, loc)).apply(myStuff)
    king.place.pantry = 100

tp = TurnProcessor(myStuff, myStuff.squads)
# ── Optimal pawn build-out ──
print("=== Optimal pawn placement ===")
for squad in myStuff.squads:
    summaries = squad.place_optimal_pawns(myStuff, tp, k=3)
    for s in summaries:
        print(f"  {squad.name}: placed pawn, king_pantry={s['king_pantry_after']}, "
              f"connected={s['connected']}")

print(f"\n{tp.status()}")

# ── A few combat/food rounds ──
print("\n=== Simulation rounds ===")
for round_num in range(5):
    moves = {sq: Move() for sq in myStuff.squads if sq.king_alive()}
    summaries = tp.play_round(moves)
    for s in summaries:
        print(f"  T{tp.turn-1} {s['squad']}: "
              f"harvested={s['food_harvested']:.0f}, diet={s['food_diet']:.0f}, "
              f"drained={s['food_drain_dealt']:.0f}, starved={s['starved']}, "
              f"king_pantry={s['king_pantry_after']}")
    if tp.game_over:
        print(f"\n🏆 Winner: {tp.winner.name if tp.winner else 'nobody'}")
        break

print(f"\n{tp.status()}")


In [ ]:
tp.turnLogDisplay( tp.summaries_by_turn)

In [ ]:
tp.turnTable( tp.summaries_by_turn)